# Kaggle launcher (THIN)

Pulls a pinned commit of [Nafish32/pulmonary](https://github.com/Nafish32/pulmonary) and runs `src.pipeline.run_all`. No business logic here.

**Before running:** click **Add data** (right panel) and attach the RSNA dataset (needs `stage_2_train_labels.csv` + `stage_2_train_images/`). VinDr is optional. Turn **GPU** on and **Internet** on in Settings for a real run.

In [ ]:
# 1) Pull pinned commit + install (idempotent: safe to re-run in the same session)
import os, subprocess

REPO_URL = "https://github.com/Nafish32/pulmonary.git"
COMMIT   = "main"   # iterate on 'main'; for a THESIS-NUMBERS run pin a tag/SHA, e.g. "thesis-run-v1"
REPO_DIR = "/kaggle/working/repo"

# CRITICAL: step out of REPO_DIR before deleting it. A prior run left the process
# CWD inside REPO_DIR; rm -rf on the CWD gives 'getcwd: cannot access parent directories'
# and every shell command after that silently no-ops.
os.chdir("/kaggle/working")

def sh(cmd):
    print(">", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd="/kaggle/working")

sh(f"rm -rf {REPO_DIR}")
sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout --quiet {COMMIT}")
sh(f"pip install -q -r {REPO_DIR}/requirements.txt")
sh(f"pip install -q -e {REPO_DIR}")

# print the exact SHA that ran, so any number is traceable even when COMMIT='main'
sha = subprocess.check_output(f"git -C {REPO_DIR} rev-parse HEAD", shell=True).decode().strip()
print("[GIT] running commit:", sha)

In [ ]:
# 2) Load + validate config (fails fast, prints fast_mode loudly)
import os, logging, sys
os.chdir("/kaggle/working/repo")
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)  # make src logs visible + live

from src.config.loader import load_config

# Smoke the chain first with fast.yaml; switch to thesis.yaml for the real numbers.
cfg = load_config("configs/fast.yaml")
# cfg = load_config("configs/thesis.yaml")
print("fast_mode =", cfg.fast_mode, "| epochs =", cfg.epochs, "| max_patients =", cfg.max_patients)

In [ ]:
# 3) Run everything end to end.
# First stages (discover rglob, DICOM caching) are IO-bound and can run minutes
# with near-zero CPU/GPU and little output -- that is normal, not a hang.
from src.pipeline import run_all
results_md = run_all(cfg)

In [ ]:
# 4) Report
print("results:", results_md)
from IPython.display import Markdown, display
with open(results_md) as f:
    display(Markdown(f.read()))

## Debug: run stages one at a time

If Cell 3 sits silent and you want to see where, interrupt it and run these. Each returns and prints, so a slow stage is obvious.

In [ ]:
from src.data.discover import discover_datasets
ds = discover_datasets(cfg.input_root); print("DISCOVER OK", ds)  # slow here == rglob over input mount

import pandas as pd
df = pd.read_csv(ds["rsna_csv"]); print("rows", len(df), "patients", df.patientId.nunique())

import torch; print("cuda:", torch.cuda.is_available())  # False -> Accelerator not actually GPU